# ARS Session 1 / Session 2 — RFT and SteerNet V_REDESIGN

Session 1 trains the current V5 architecture. Session 2 reuses saved Session 1 rollouts to train and evaluate the V2 redesign.


## 0. Configure Kaggle runtime and checkout

Set allocator and tokenizer environment variables before launching Python worker processes. The checkout is refreshed from the `development` branch so notebook commands match the committed code.

## Execution and scale configuration

`SESSION=1` runs the original V5 flow. `SESSION=2` trains V_REDESIGN on saved rollouts when available and automatically collects fresh rollouts if they are missing. Phase failures are recorded and dependent steps are skipped so the notebook reaches its final status report.


In [ ]:
# ==========================================
# EXECUTION & SCALE CONFIGURATION
# ==========================================
import os
from datetime import datetime
from pathlib import Path
SESSION = 2  # 1 = original V5 flow; 2 = V_REDESIGN on saved rollouts
SMOKE_TEST = False  # Set to True for a 5-min sanity run, False for production run
ATTENTION_IMPLEMENTATION = "sdpa"  # Eager T4 probe produced invalid sampled logits
os.environ["ARS_ATTENTION_IMPLEMENTATION"] = ATTENTION_IMPLEMENTATION
# Set this to an existing report/rft/<run_id> to resume from collected shards.
RESUME_RFT_RUN_ID = None  # Session 1 resume only
SESSION2_ROLLOUT_DIR = Path("report/rft/20261116_134631")  # Mount/copy saved shards here or set /kaggle/input/...
SESSION1_CHECKPOINT = Path("checkpoints/steernet.pt")  # Attach Session 1 V5 checkpoint to compare on the identical 800 rows
SESSION2_CHECKPOINT = Path("checkpoints/steernet_v2.pt")
SESSION2_NORM_CALIBRATION = Path("report/session2/norms_v2.json")
SESSION2_RFT_DIR = Path("report/session2/rft")
SESSION2_BENCH_N_TEST = 800  # Same deterministic shuffled test order as Session 1
SESSION2_BENCH_SEED = BENCH_SEED if "BENCH_SEED" in globals() else 42
RUN_ID = RESUME_RFT_RUN_ID or datetime.now().strftime("%Y%m%d_%H%M%S")
if RESUME_RFT_RUN_ID is not None:
    resume_dir = Path("report/rft") / RESUME_RFT_RUN_ID
    required_resume_files = [
        resume_dir / "gate0_stats.json",
        resume_dir / "shard0.jsonl",
        resume_dir / "shard0_stats.json",
        resume_dir / "shard1.jsonl",
        resume_dir / "shard1_stats.json",
    ]
    missing_resume_files = [path.name for path in required_resume_files if not path.is_file()]
    if missing_resume_files:
        print(
            f"Saved run {RESUME_RFT_RUN_ID} is unavailable in this Kaggle session "
            f"(missing: {missing_resume_files}); starting a fresh Session 1 run.",
            flush=True,
        )
        RESUME_RFT_RUN_ID = None
        RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PHASE_STATUS = {
    "equivalence": "pending", "gate0": "pending",
    "collection": "pending", "training": "pending",
    "calibration": "pending", "benchmark": "pending",
}
GATE0_N_OVERRIDE = None  # Optional explicit Gate 0 size; set to 60 to run the full probe in smoke mode

if SMOKE_TEST:
    EQUIVALENCE_N = 10        # Two quick passes over a small probe
    GATE0_N = 10              # Small Gate 0 sample; threshold is diagnostic in smoke mode
    N_RFT_QUESTIONS = 40       # Quick sample across shards
    RFT_K = 2                  # Rollouts per question
    TRAIN_EPOCHS = 1           # 1 epoch for fast sanity
    CALIB_N = 16               # Small calibration sample
    BENCH_N_TEST = 30          # Small benchmark test subset
    BENCH_SHOTS = ["0"]        # Fast single condition
else:
    EQUIVALENCE_N = 100       # Full equivalence probe
    GATE0_N = 60               # Production Gate 0 sample
    N_RFT_QUESTIONS = 2000     # Full production sample
    RFT_K = 8                  # Full rollouts
    TRAIN_EPOCHS = 2           # Full training epochs
    CALIB_N = 128              # Standard calibration sample
    BENCH_N_TEST = 1319        # Full test set
    BENCH_SHOTS = ["0", "2"] # Full 0-shot and 2-shot
BENCH_SEED = 42            # Deterministic shuffled subset of the full test split

if GATE0_N_OVERRIDE is not None:
    if GATE0_N_OVERRIDE < 1:
        raise ValueError("GATE0_N_OVERRIDE must be positive")
    GATE0_N = int(GATE0_N_OVERRIDE)

print({
    "SESSION": SESSION,
    "SESSION2_ROLLOUT_DIR": str(SESSION2_ROLLOUT_DIR),
    "SESSION2_BENCH_N_TEST": SESSION2_BENCH_N_TEST,
    "SMOKE_TEST": SMOKE_TEST,
    "RESUME_RFT_RUN_ID": RESUME_RFT_RUN_ID,
    "RUN_ID": RUN_ID,
    "EQUIVALENCE_N": EQUIVALENCE_N,
    "GATE0_N": GATE0_N,
    "GATE0_N_OVERRIDE": GATE0_N_OVERRIDE,
    "N_RFT_QUESTIONS": N_RFT_QUESTIONS,
    "RFT_K": RFT_K,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "CALIB_N": CALIB_N,
    "BENCH_N_TEST": BENCH_N_TEST,
    "BENCH_SEED": BENCH_SEED,
    "BENCH_SHOTS": BENCH_SHOTS,
})

In [ ]:
import codecs
import json
import os
import selectors
import subprocess
import sys
import time
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"

REPO_URL = "https://github.com/Abdullah182155/adaptive-residual-steering.git"
REPO_DIR = Path("/kaggle/working/adaptive-residual-steering")
BRANCH = "development"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print(f"Repository: {REPO_DIR} ({BRANCH})", flush=True)

def _emit_stream_line(state, line, *, force=False):
    """Print child output snapshots as normal notebook lines with a short throttle."""
    now = time.monotonic()
    if line and (force or now - state["last_update"] >= 1.0 or "100%" in line):
        print(f"{state['prefix']}{line}", flush=True)
        state["last_update"] = now


def _consume_stream_chunk(state, chunk):
    """Split child output on newline and tqdm carriage-return boundaries."""
    state["pending"] += state["decoder"].decode(chunk)
    while "\r" in state["pending"] or "\n" in state["pending"]:
        cr = state["pending"].find("\r") if "\r" in state["pending"] else len(state["pending"])
        lf = state["pending"].find("\n") if "\n" in state["pending"] else len(state["pending"])
        boundary = min(cr, lf)
        separator = state["pending"][boundary]
        line, state["pending"] = state["pending"][:boundary], state["pending"][boundary + 1:]
        _emit_stream_line(state, line, force=(separator == "\n" or "100%" in line))


def _finish_stream(state):
    state["pending"] += state["decoder"].decode(b"", final=True)
    _emit_stream_line(state, state["pending"], force=True)


def _start_live_process(command, env=None):
    return subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )


def run_live(command, *, env=None, prefix=""):
    """Run one child and render its tqdm refreshes as flushed notebook output."""
    run_parallel_live([(prefix, command, env)])


def run_parallel_live(jobs):
    """Run child commands concurrently and stream each labelled pipe in this thread."""
    selector = selectors.DefaultSelector()
    processes = []
    for prefix, command, env in jobs:
        process = _start_live_process(command, env)
        processes.append((prefix, command, process))
        state = {
            "prefix": prefix,
            "decoder": codecs.getincrementaldecoder("utf-8")(errors="replace"),
            "pending": "",
            "last_update": 0.0,
        }
        selector.register(process.stdout, selectors.EVENT_READ, state)

    while selector.get_map():
        for key, _ in selector.select(timeout=0.25):
            chunk = os.read(key.fileobj.fileno(), 4096)
            if chunk:
                _consume_stream_chunk(key.data, chunk)
            else:
                _finish_stream(key.data)
                selector.unregister(key.fileobj)
                key.fileobj.close()

    selector.close()
    failures = []
    for prefix, command, process in processes:
        return_code = process.wait()
        if return_code:
            failures.append(f"{prefix.strip()} exited with code {return_code}: {command}")
    if failures:
        raise RuntimeError("\n".join(failures))


## 1. Verify the Kaggle accelerator and repository tests

Check the runtime and test suite, but record failures as diagnostics rather than aborting the notebook. The collection stage will report whether one or both T4 workers were available.


In [ ]:
try:
    gpu_query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        check=True, capture_output=True, text=True,
    )
    gpu_rows = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]
    print("Visible GPUs:", gpu_rows, flush=True)
    if len(gpu_rows) < 2 or any("T4" not in row for row in gpu_rows[:2]):
        print("GPU diagnostic: Dual T4 was not detected; dependent work may fail and will be recorded.", flush=True)
except Exception as exc:
    gpu_rows = []
    print(f"GPU diagnostic unavailable: {type(exc).__name__}: {exc}", flush=True)

try:
    test_process = subprocess.run(
        [sys.executable, "-u", "-m", "pytest", "tests/", "-x", "-q"],
        check=False,
    )
    PHASE_STATUS["tests"] = "passed" if test_process.returncode == 0 else f"failed_exit_{test_process.returncode}"
except Exception as exc:
    PHASE_STATUS["tests"] = f"failed: {type(exc).__name__}: {exc}"
print("Test status:", PHASE_STATUS.get("tests"), flush=True)


## 2. Check the active session configuration

Validate the locked Stage 1 or V_REDESIGN settings. Any configuration issue is written into the final status summary and prevents dependent work from launching.


In [ ]:
from src.configs.config import RSCConfig

CONFIG_VALID = False
try:
    cfg = RSCConfig(steernet_arch="v2" if SESSION == 2 else "v5")
    assert SESSION in (1, 2)
    if SESSION == 1:
        assert cfg.fixed_layers == [17, 20, 23, 26]
        assert cfg.steernet_use_mag_head is False
        assert cfg.use_gate is False
    else:
        assert cfg.fixed_layers == [12, 15, 18, 21]
        assert cfg.lora_rank == 16 and cfg.rho_max == 0.10
        assert cfg.steernet_norm_stat == "p99_norm"
        assert cfg.steernet_use_mag_head is False and cfg.steernet_use_conv is False
        cfg.layer_norms_calibration = str(SESSION2_NORM_CALIBRATION)
    assert cfg.phase1_lr == 1.5e-4
    assert cfg.objective == "nll"
    assert cfg.max_tok_len == 768
    assert cfg.batch_size == 2 and cfg.grad_accum == 16
    assert cfg.generation_repetition_penalty == 1.0
    cfg.validate()
    CONFIG_VALID = True
    print({
        "session": SESSION, "architecture": cfg.steernet_arch,
        "route": cfg.fixed_layers, "rank": cfg.lora_rank,
        "rho_max": cfg.rho_max, "norm_reference": cfg.steernet_norm_stat,
        "mag_head": cfg.steernet_use_mag_head,
        "loss": "positive completion token-mean NLL",
        "phase1_lr": cfg.phase1_lr, "use_gate": cfg.use_gate,
    }, flush=True)
except Exception as exc:
    PHASE_STATUS["configuration"] = f"failed: {type(exc).__name__}: {exc}"
    print("Configuration diagnostic failed; dependent phases will be skipped:", exc, flush=True)


## Session 2 — V_REDESIGN (reuse Session 1 rollouts)



Attach or copy the saved `shard0.jsonl` and `shard1.jsonl` into `SESSION2_ROLLOUT_DIR`. This path never launches collection. It calibrates p99 norms from those shard questions, trains SteerNetV2, calibrates the deployment contract, then evaluates `steer_only` against baseline on the fixed shuffled benchmark subset.

In [ ]:
if SESSION == 2 and CONFIG_VALID:
    SESSION2_STATUS = {
        "rollouts": "pending", "norm_calibration": "pending",
        "training": "pending", "deployment_calibration": "pending",
        "benchmark": "pending", "v5_comparison": "pending",
    }
    SESSION2_BENCH_N_TEST = 800
    SESSION2_BENCH_SEED = BENCH_SEED
    SESSION2_RFT_DIR.mkdir(parents=True, exist_ok=True)
    SESSION2_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

    session2_env = os.environ.copy()
    session2_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    session2_env["TOKENIZERS_PARALLELISM"] = "false"
    session2_env["PYTHONUNBUFFERED"] = "1"
    session2_env["CUDA_VISIBLE_DEVICES"] = "0"

    def session2_run(phase, command, *, env=None, prefix=""):
        """Run one Session 2 phase and record failures without stopping the notebook."""
        print(f"Starting Session 2 {phase}", flush=True)
        try:
            run_live(command, env=env, prefix=prefix)
            SESSION2_STATUS[phase] = "passed"
            return True
        except Exception as exc:
            SESSION2_STATUS[phase] = f"failed: {type(exc).__name__}: {exc}"
            print(f"Session 2 {phase} failed; dependent phases will be skipped: {exc}", flush=True)
            return False

    def _nonempty_file(path):
        """Return false for missing, empty, or temporarily inaccessible artifacts."""
        try:
            return path.is_file() and path.stat().st_size > 0
        except OSError:
            return False

    def valid_rollout_directory(directory):
        """Accept a saved pair of non-empty question rollout shards."""
        return all(_nonempty_file(directory / f"shard{i}.jsonl") for i in (0, 1))

    def completed_new_shard(directory, shard):
        """A newly collected shard is complete only when its stats were written."""
        return (
            _nonempty_file(directory / f"shard{shard}.jsonl")
            and _nonempty_file(directory / f"shard{shard}_stats.json")
        )

    rollout_candidates = [SESSION2_ROLLOUT_DIR, Path("report/rft")]
    rollout_candidates.extend(
        path.parent for path in Path("report/rft").glob("*/shard0.jsonl")
    )
    rollout_candidates.extend(
        path.parent for path in SESSION2_RFT_DIR.glob("rollouts_*/shard*.jsonl")
    )
    rollout_candidates = list(dict.fromkeys(rollout_candidates))
    rollout_candidates.sort(
        key=lambda item: max(
            (child.stat().st_mtime for child in item.glob("shard*.jsonl") if child.exists()),
            default=0,
        ),
        reverse=True,
    )
    existing_rollout_dir = next(
        (candidate for candidate in rollout_candidates if valid_rollout_directory(candidate)),
        None,
    )
    partial_rollout_dir = next(
        (
            candidate for candidate in rollout_candidates
            if any(completed_new_shard(candidate, shard) for shard in (0, 1))
            and _nonempty_file(candidate / "gate0_stats.json")
        ),
        None,
    )

    if existing_rollout_dir is not None:
        SESSION2_ACTIVE_ROLLOUT_DIR = existing_rollout_dir
        SESSION2_STATUS["rollouts"] = "reused"
        print(f"Reusing saved rollout shards from {SESSION2_ACTIVE_ROLLOUT_DIR}; collection skipped.", flush=True)
    else:
        if partial_rollout_dir is not None:
            SESSION2_ACTIVE_ROLLOUT_DIR = partial_rollout_dir
            print(
                f"Resuming partial collection in {SESSION2_ACTIVE_ROLLOUT_DIR}; "
                "completed shards will be kept.", flush=True,
            )
        else:
            SESSION2_ACTIVE_ROLLOUT_DIR = SESSION2_RFT_DIR / f"rollouts_{RUN_ID}"
            SESSION2_ACTIVE_ROLLOUT_DIR.mkdir(parents=True, exist_ok=True)
            print(
                "No saved rollout shards found. Collecting fresh data into "
                f"{SESSION2_ACTIVE_ROLLOUT_DIR}.", flush=True,
            )

        gate_stats_path = SESSION2_ACTIVE_ROLLOUT_DIR / "gate0_stats.json"
        if _nonempty_file(gate_stats_path):
            try:
                gate_stats = json.loads(gate_stats_path.read_text(encoding="utf-8"))
                gate_ok = bool(gate_stats.get("passed", False))
                SESSION2_STATUS["gate0"] = "reused" if gate_ok else "diagnostic_failed_nonblocking"
            except (OSError, ValueError) as exc:
                print(f"Saved Gate 0 stats unreadable; rerunning Gate 0: {exc}", flush=True)
                gate_ok = False
        else:
            gate_ok = False

        if not gate_ok:
            gate_env = session2_env.copy()
            gate_env["CUDA_VISIBLE_DEVICES"] = "0"
            gate_ok = session2_run(
                "gate0", [
                    sys.executable, "-u", "scripts/run_rft_pipeline.py",
                    "--stage", "gate0", "--n_questions", str(GATE0_N),
                    "--k", "8", "--temperature", "0.7",
                    "--gate0_nonblocking", "--output_dir", str(SESSION2_ACTIVE_ROLLOUT_DIR),
                ], env=gate_env, prefix="[S2/gate0] ",
            )

        if gate_ok:
            for attempt in range(2):
                pending_shards = [
                    shard for shard in (0, 1)
                    if not completed_new_shard(SESSION2_ACTIVE_ROLLOUT_DIR, shard)
                ]
                if not pending_shards:
                    SESSION2_STATUS["rollouts"] = "collected_or_resumed"
                    break
                workers = []
                for shard in pending_shards:
                    # First attempt uses each shard's mapped T4; retry uses GPU0 serially.
                    gpu = str(shard) if attempt == 0 else "0"
                    worker_env = session2_env.copy()
                    worker_env["CUDA_VISIBLE_DEVICES"] = gpu
                    workers.append((
                        f"[S2/GPU{gpu}/shard{shard}] ",
                        [
                            sys.executable, "-u", "scripts/run_rft_pipeline.py",
                            "--stage", "collect",
                            "--n_questions", str(N_RFT_QUESTIONS),
                            "--k", str(RFT_K), "--temperature", "0.7",
                            "--n_shards", "2", "--shard", str(shard),
                            "--allow_low_yield",
                            "--output_dir", str(SESSION2_ACTIVE_ROLLOUT_DIR),
                        ],
                        worker_env,
                    ))
                try:
                    run_parallel_live(workers)
                except Exception as exc:
                    print(
                        f"Collection attempt {attempt + 1} had a worker error; "
                        f"checking saved shards before retry: {exc}", flush=True,
                    )
                pending_after = [
                    shard for shard in (0, 1)
                    if not completed_new_shard(SESSION2_ACTIVE_ROLLOUT_DIR, shard)
                ]
                if not pending_after:
                    SESSION2_STATUS["rollouts"] = "collected_or_resumed"
                    break
                SESSION2_STATUS["rollouts"] = f"incomplete_shards: {pending_after}"
            else:
                print(
                    "Collection remains partial after retry; keeping completed shard files for training.",
                    flush=True,
                )
        else:
            SESSION2_STATUS["rollouts"] = "failed_gate0_worker"

    rollout_files_for_training = [
        SESSION2_ACTIVE_ROLLOUT_DIR / f"shard{i}.jsonl" for i in (0, 1)
        if _nonempty_file(SESSION2_ACTIVE_ROLLOUT_DIR / f"shard{i}.jsonl")
        and (
            SESSION2_STATUS["rollouts"] == "reused"
            or completed_new_shard(SESSION2_ACTIVE_ROLLOUT_DIR, i)
        )
    ]
    rollouts_ready = bool(rollout_files_for_training)
    if rollouts_ready and CONFIG_VALID:
        if len(rollout_files_for_training) < 2:
            SESSION2_STATUS["rollouts"] = "partial_data_used_for_training"
            print(
                "Only one completed shard is available; continuing with its verified positives "
                "so training and the 800-question benchmark can still run.", flush=True,
            )
        SESSION2_NORM_CALIBRATION.parent.mkdir(parents=True, exist_ok=True)
        calibrated = session2_run(
            "norm_calibration", [
                sys.executable, "-u", "scripts/calibrate_norms.py",
                "--n_samples", str(CALIB_N),
                "--candidate_layers", "12", "15", "18", "21",
                "--max_length", "768", "--seed", str(BENCH_SEED),
                "--rollout_files",
                *(str(path) for path in rollout_files_for_training),
                "--output", str(SESSION2_NORM_CALIBRATION),
            ], env=session2_env, prefix="[S2/p99] ",
        )
        if calibrated:
            try:
                norm_report = json.loads(SESSION2_NORM_CALIBRATION.read_text(encoding="utf-8"))
                for layer in (12, 15, 18, 21):
                    if norm_report["layers"][str(layer)]["p99_norm"] <= 0:
                        raise ValueError(f"layer {layer} has a non-positive p99 norm")
            except Exception as exc:
                SESSION2_STATUS["norm_calibration"] = f"failed: {type(exc).__name__}: {exc}"
                print("Norm calibration validation failed:", exc, flush=True)
                calibrated = False

        if calibrated:
            trained = session2_run(
                "training", [
                    sys.executable, "-u", "scripts/run_rft_pipeline.py",
                    "--stage", "train", "--steernet_arch", "v2",
                    "--skip_collection",
                    "--rollout_dir", str(SESSION2_ACTIVE_ROLLOUT_DIR),
                    *( ["--allow_partial_rollouts"] if len(rollout_files_for_training) < 2 else [] ),
                    "--norm_calibration", str(SESSION2_NORM_CALIBRATION),
                    "--epochs", str(TRAIN_EPOCHS),
                    "--checkpoint", str(SESSION2_CHECKPOINT),
                    "--output_dir", str(SESSION2_RFT_DIR),
                ], env=session2_env, prefix="[S2/train] ",
            )
            trained = trained and SESSION2_CHECKPOINT.is_file()
            if not trained and SESSION2_STATUS["training"] == "passed":
                SESSION2_STATUS["training"] = f"failed: checkpoint not written at {SESSION2_CHECKPOINT}"

        if SESSION2_CHECKPOINT.is_file() and SESSION2_STATUS["training"] == "passed":
            contract_path = Path("report/session2/deployment_contract_v2.json")
            bench_record = Path(
                f"report/session2/gsm8k_{SESSION2_BENCH_N_TEST}_seed{SESSION2_BENCH_SEED}_v2.json"
            )
            common_overrides = [
                "--set", "lora_rank=16", "--set", "rho_max=0.1",
                "--set", "steernet_norm_stat=p99_norm",
                "--set", "steernet_use_mag_head=false",
                "--set", "steernet_use_conv=false",
                "--set", f"layer_norms_calibration={SESSION2_NORM_CALIBRATION}",
                "--set", f"lora_save_path={SESSION2_CHECKPOINT}",
            ]
            s5_ok = session2_run(
                "deployment_calibration", [
                    sys.executable, "-u", "scripts/run_ars_pipeline.py",
                    "--stage", "S5", "--steernet_arch", "v2",
                    "--s_fixed", "12,15,18,21",
                    "--calib_sel_n", str(CALIB_N),
                    "--contract", str(contract_path), *common_overrides,
                ], env=session2_env, prefix="[S2/S5] ",
            )
            if s5_ok:
                Path("report/session2").mkdir(parents=True, exist_ok=True)
                bench_ok = session2_run(
                    "benchmark", [
                        sys.executable, "-u", "scripts/run_ars_pipeline.py",
                        "--stage", "bench", "--steernet_arch", "v2",
                        "--s_fixed", "12,15,18,21",
                        "--contract", str(contract_path),
                        "--n_test", str(SESSION2_BENCH_N_TEST),
                        "--test_seed", str(SESSION2_BENCH_SEED),
                        "--shots", "0", "2",
                        "--arms", "baseline,steer_only",
                        "--record", str(bench_record), "--report", *common_overrides,
                    ], env=session2_env, prefix="[S2/benchmark] ",
                )
                if bench_ok:
                    try:
                        result = json.loads(bench_record.read_text(encoding="utf-8"))
                        for shots in ("0", "2"):
                            observed_n = result["shots"][shots]["baseline"]["n"]
                            if observed_n != SESSION2_BENCH_N_TEST:
                                raise ValueError(
                                    f"shots={shots}: expected n={SESSION2_BENCH_N_TEST}, got {observed_n}"
                                )
                            paired = result["shots"][shots]["paired_steer_only"]
                            print({"shots": shots, **paired}, flush=True)
                    except Exception as exc:
                        SESSION2_STATUS["benchmark"] = f"failed: {type(exc).__name__}: {exc}"
                        print("Benchmark result could not be validated:", exc, flush=True)
                    print("Session 2 artifacts:", SESSION2_CHECKPOINT, SESSION2_NORM_CALIBRATION,
                          contract_path, bench_record, flush=True)
    else:
        for phase in ("norm_calibration", "training", "deployment_calibration", "benchmark"):
            if SESSION2_STATUS[phase] == "pending":
                SESSION2_STATUS[phase] = "skipped: valid rollouts or configuration unavailable"

    for phase in ("norm_calibration", "training", "deployment_calibration", "benchmark"):
        if SESSION2_STATUS[phase] == "pending":
            SESSION2_STATUS[phase] = "skipped: prerequisite phase did not pass"

    # Optional same-subset comparison against the Session 1 V5 checkpoint.
    if SESSION1_CHECKPOINT.is_file():
        v5_contract = Path("report/session2/deployment_contract_v5.json")
        v5_record = Path(
            f"report/session2/gsm8k_{SESSION2_BENCH_N_TEST}_seed{SESSION2_BENCH_SEED}_v5.json"
        )
        v5_overrides = ["--set", f"lora_save_path={SESSION1_CHECKPOINT}"]
        v5_s5 = session2_run(
            "v5_comparison", [
                sys.executable, "-u", "scripts/run_ars_pipeline.py",
                "--stage", "S5", "--steernet_arch", "v5",
                "--calib_sel_n", str(CALIB_N),
                "--contract", str(v5_contract), *v5_overrides,
            ], env=session2_env, prefix="[S1/V5-S5] ",
        )
        if v5_s5:
            v5_bench_ok = session2_run(
                "v5_comparison", [
                    sys.executable, "-u", "scripts/run_ars_pipeline.py",
                    "--stage", "bench", "--steernet_arch", "v5",
                    "--contract", str(v5_contract),
                    "--n_test", str(SESSION2_BENCH_N_TEST),
                    "--test_seed", str(SESSION2_BENCH_SEED),
                    "--shots", "0", "2",
                    "--arms", "baseline,steer_only",
                    "--record", str(v5_record), "--report",
                    *v5_overrides,
                ], env=session2_env, prefix="[S1/V5-benchmark] ",
            )
            if v5_bench_ok:
                try:
                    v5_result = json.loads(v5_record.read_text(encoding="utf-8"))
                    for shots in ("0", "2"):
                        paired = v5_result["shots"][shots]["paired_steer_only"]
                        print({"architecture": "v5", "shots": shots, **paired}, flush=True)
                    SESSION2_STATUS["v5_comparison"] = "passed"
                except Exception as exc:
                    SESSION2_STATUS["v5_comparison"] = f"failed: {type(exc).__name__}: {exc}"
                    print("V5 comparison result could not be validated:", exc, flush=True)
    else:
        SESSION2_STATUS["v5_comparison"] = (
            f"skipped: attach the Session 1 checkpoint at {SESSION1_CHECKPOINT} "
            "to compare V5 on the same deterministic 800-question subset"
        )
        print(SESSION2_STATUS["v5_comparison"], flush=True)

    print("Session 2 phase status:", json.dumps(SESSION2_STATUS, indent=2), flush=True)
elif SESSION == 2:
    print("Session 2 skipped: configuration validation failed.", flush=True)


## Session 1 — Step 1 — Batched vs. batch-1 generation equivalence

This creates `report/rft/generation_equivalence.json`. It compares B=2 with batch-1 as a diagnostic. Session 1 collection and benchmark both generate one question at a time, so this unused batched path does not block them. If you later enable batched evaluation, require this check to pass first.

In [ ]:
if SESSION == 1:
    RFT_DIR = Path("report/rft") / RUN_ID
    RFT_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME_RFT_RUN_ID is not None:
        PHASE_STATUS["equivalence"] = "skipped_resume"
        print(f"Resuming saved RFT run {RUN_ID}; skipping equivalence rerun.", flush=True)
    else:
        print(f"Starting equivalence probe: {EQUIVALENCE_N} questions", flush=True)
        equivalence_command = [
            sys.executable, "-u", "scripts/run_rft_pipeline.py",
            "--stage", "equivalence",
            "--n_questions", str(EQUIVALENCE_N),
            "--output_dir", str(RFT_DIR),
        ]
        try:
            run_live(equivalence_command, prefix="[equivalence] ")
            equivalence = json.loads((RFT_DIR / "generation_equivalence.json").read_text(encoding="utf-8"))
            PHASE_STATUS["equivalence"] = "passed" if equivalence["quality_gate_passed"] else "diagnostic_mismatch"
        except (RuntimeError, OSError, ValueError, KeyError) as exc:
            PHASE_STATUS["equivalence"] = f"failed: {exc}"
        print("Equivalence status:", PHASE_STATUS["equivalence"], flush=True)


## Session 1 — Step 2 — Gate 0 sampling sanity probe

Gate 0 compares 8 sampled continuations at temperature 0.7 with paired greedy decoding on `GATE0_N` questions. The notebook records whether the 70% and paired-greedy criteria pass. If they fail, dependent RFT steps are marked skipped and the notebook continues to the final status cell.

In [ ]:
if SESSION == 1:
    if RESUME_RFT_RUN_ID is not None:
        GATE0_OK = True
        PHASE_STATUS["gate0"] = "skipped_resume"
        print(f"Resuming saved RFT run {RUN_ID}; Gate 0 was already completed.", flush=True)
    else:
        print(f"Starting Gate 0 probe on {GATE0_N} questions", flush=True)
        gate0_command = [
            sys.executable, "-u", "scripts/run_rft_pipeline.py",
            "--stage", "gate0",
            "--n_questions", str(GATE0_N),
            "--k", "8",
            "--temperature", "0.7",
            "--output_dir", str(RFT_DIR),
        ]
        gate0_command.append("--gate0_nonblocking")
        try:
            run_live(gate0_command, prefix="[gate0] ")
            gate0 = json.loads((RFT_DIR / "gate0_stats.json").read_text(encoding="utf-8"))
            GATE0_OK = bool(gate0["quality_gate_passed"])
            PHASE_STATUS["gate0"] = "passed" if GATE0_OK else "quality_failed"
            print({key: gate0[key] for key in ("n", "pass_at_k_lenient", "greedy_lenient_accuracy", "pass_at_k_delta_vs_greedy_pp", "quality_gate_passed", "passed")})
        except (RuntimeError, OSError, ValueError) as exc:
            GATE0_OK = False
            PHASE_STATUS["gate0"] = f"failed: {exc}"
            print("Gate 0 could not complete; dependent RFT stages will be skipped:", exc, flush=True)
        if not GATE0_OK:
            print("Gate 0 did not meet its quality threshold; collection and training are skipped.", flush=True)


## Session 1 — Step 3 — Collect positive RFT rollouts on both T4s

Each worker receives a disjoint shard of the same seeded configured-question sample and writes its own JSONL/stat files. Workers are launched as unbuffered processes with live output inherited by the Kaggle cell, so both per-shard progress bars remain visible. Each process sees one T4 and initializes CUDA after device selection. The notebook reports the combined verified-positive yield; production skips training below 15%, while smoke mode may train only when at least one verified positive exists.

In [ ]:
if SESSION == 1:
    COLLECTION_OK = False
    RFT_ROWS_READY = False
    if RESUME_RFT_RUN_ID is not None:
        try:
            shard_stats = [json.loads((RFT_DIR / f"shard{i}_stats.json").read_text(encoding="utf-8")) for i in (0, 1)]
            total_rollouts = 0
            positive_rollouts = 0
            for shard in (0, 1):
                with (RFT_DIR / f"shard{shard}.jsonl").open(encoding="utf-8") as stream:
                    for line in stream:
                        record = json.loads(line)
                        total_rollouts += len(record["samples"])
                        positive_rollouts += sum(bool(sample.get("strict_ok")) for sample in record["samples"])
            positive_rate = positive_rollouts / max(total_rollouts, 1)
            RFT_ROWS_READY = positive_rollouts > 0 and positive_rate >= 0.15
            COLLECTION_OK = True
            PHASE_STATUS["collection"] = "resumed_ready" if RFT_ROWS_READY else "insufficient_positive_yield"
            print({"resumed_run": RUN_ID, "shard0": shard_stats[0], "shard1": shard_stats[1], "positive_rollouts": positive_rollouts, "positive_rate": positive_rate, "training_ready": RFT_ROWS_READY}, flush=True)
        except (OSError, ValueError, KeyError) as exc:
            PHASE_STATUS["collection"] = f"resume_failed: {exc}"
            print("Saved rollout shards could not be resumed; training will be skipped:", exc, flush=True)
    elif not GATE0_OK:
        PHASE_STATUS["collection"] = "skipped_gate0_failed"
        print("Skipping rollout collection because Gate 0 did not pass.", flush=True)
    else:
        collection_env = os.environ.copy()
        collection_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        collection_env["TOKENIZERS_PARALLELISM"] = "false"
        collection_env["PYTHONUNBUFFERED"] = "1"
        workers = []
        for shard, gpu in ((0, "0"), (1, "1")):
            env = collection_env.copy()
            env["CUDA_VISIBLE_DEVICES"] = gpu
            command = [
                sys.executable, "-u", "scripts/run_rft_pipeline.py",
                "--stage", "collect",
                "--n_questions", str(N_RFT_QUESTIONS),
                "--k", str(RFT_K),
                "--temperature", "0.7",
                "--n_shards", "2",
                "--shard", str(shard),
                "--output_dir", str(RFT_DIR),
                "--allow_low_yield",
            ]
            workers.append((f"[GPU{gpu}/shard{shard}] ", command, env))
        try:
            print("Starting both rollout shards; live progress follows:", flush=True)
            run_parallel_live(workers)
            shard_stats = [json.loads((RFT_DIR / f"shard{i}_stats.json").read_text(encoding="utf-8")) for i in (0, 1)]
            total_rollouts = 0
            positive_rollouts = 0
            for shard in (0, 1):
                with (RFT_DIR / f"shard{shard}.jsonl").open(encoding="utf-8") as stream:
                    for line in stream:
                        record = json.loads(line)
                        total_rollouts += len(record["samples"])
                        positive_rollouts += sum(bool(sample.get("strict_ok")) for sample in record["samples"])
            positive_rate = positive_rollouts / max(total_rollouts, 1)
            RFT_ROWS_READY = positive_rollouts > 0 and (SMOKE_TEST or positive_rate >= 0.15)
            COLLECTION_OK = True
            PHASE_STATUS["collection"] = "ready" if RFT_ROWS_READY else "insufficient_positive_yield"
            print({"shard0": shard_stats[0], "shard1": shard_stats[1], "positive_rollouts": positive_rollouts, "positive_rate": positive_rate, "training_ready": RFT_ROWS_READY})
        except (RuntimeError, OSError, ValueError, KeyError) as exc:
            PHASE_STATUS["collection"] = f"failed: {exc}"
            print("Collection could not complete; training will be skipped:", exc, flush=True)


## Session 1 — Step 4 — Train Stage 1 RFT

Training combines the two rollout shards and samples mixed 0/2/4-shot contexts. It uses the locked fixed route, identity magnitude, frozen Phi-2, and positive-completion token-mean NLL. The trainer uses micro-batch 2 with gradient accumulation 16, gradient checkpointing, and a maximum 768-token sequence length. The resulting adapter is `checkpoints/steernet.pt`.

In [ ]:
if SESSION == 1:
    TRAIN_OK = False
    if not RFT_ROWS_READY:
        PHASE_STATUS["training"] = "skipped_no_usable_positive_rollouts"
        print("Skipping RFT training: no positives or production yield is below 15%.", flush=True)
    else:
        train_env = os.environ.copy()
        train_env["CUDA_VISIBLE_DEVICES"] = "0"
        train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
        train_env["TOKENIZERS_PARALLELISM"] = "false"
        print(f"Starting RFT training for {TRAIN_EPOCHS} epoch(s)", flush=True)
        try:
            run_live([
                sys.executable, "-u", "scripts/run_rft_pipeline.py",
                "--stage", "train",
                "--epochs", str(TRAIN_EPOCHS),
                "--checkpoint", "checkpoints/steernet.pt",
                "--output_dir", str(RFT_DIR),
            ], env=train_env, prefix="[train] ")
            TRAIN_OK = Path("checkpoints/steernet.pt").is_file()
            PHASE_STATUS["training"] = "passed" if TRAIN_OK else "failed_no_checkpoint"
        except (RuntimeError, OSError, ValueError) as exc:
            PHASE_STATUS["training"] = f"failed: {exc}"
            print("Training failed; calibration and steering benchmark will be skipped:", exc, flush=True)

## Session 1 — Step 5 — Calibrate and evaluate pure steering vs. baseline

The deployment contract is calibrated from validation data using the trained checkpoint. The paired benchmark then evaluates baseline and the isolated `steer_only` arm on the configured GSM8K test subset and shot conditions. Evaluation is greedy, uses repetition penalty 1.0, one question per generation, and reloads the dedicated `steernet.pt` checkpoint. No gate/router arm is enabled.

In [ ]:
if SESSION == 1:
    BENCH_RECORD = Path("report") / f"phase1_gsm8k_{BENCH_N_TEST}_{RUN_ID}.json"
    CONTRACT_PATH = Path("report") / f"deployment_contract_{RUN_ID}.json"
    BENCH_OK = False
    CALIBRATION_OK = False
    eval_env = os.environ.copy()
    eval_env["CUDA_VISIBLE_DEVICES"] = "0"
    eval_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    eval_env["TOKENIZERS_PARALLELISM"] = "false"
    if not TRAIN_OK:
        PHASE_STATUS["calibration"] = "skipped_no_trained_checkpoint"
        PHASE_STATUS["benchmark"] = "skipped_no_trained_checkpoint"
        print("Skipping calibration and steering benchmark because training did not produce a checkpoint.", flush=True)
    else:
        try:
            print(f"Starting S5 calibration on {CALIB_N} rows", flush=True)
            run_live([
                sys.executable, "-u", "scripts/run_ars_pipeline.py",
                "--stage", "S5",
                "--s_fixed", "17,20,23,26",
                "--calib_sel_n", str(CALIB_N),
                "--contract", str(CONTRACT_PATH),
            ], env=eval_env, prefix="[calibration] ")
            CALIBRATION_OK = CONTRACT_PATH.is_file()
            PHASE_STATUS["calibration"] = "passed" if CALIBRATION_OK else "failed_no_contract"
        except (RuntimeError, OSError, ValueError) as exc:
            PHASE_STATUS["calibration"] = f"failed: {exc}"
            print("Calibration failed; benchmark will be skipped:", exc, flush=True)
        if CALIBRATION_OK:
            try:
                print(f"Starting benchmark: n={BENCH_N_TEST}, shots={BENCH_SHOTS}", flush=True)
                run_live([
                    sys.executable, "-u", "scripts/run_ars_pipeline.py",
                    "--stage", "bench",
                    "--s_fixed", "17,20,23,26",
                    "--contract", str(CONTRACT_PATH),
                    "--n_test", str(BENCH_N_TEST),
                    "--test_seed", str(BENCH_SEED),
                    "--shots", *BENCH_SHOTS,
                    "--arms", "baseline,steer_only",
                    "--record", str(BENCH_RECORD),
                    "--report",
                ], env=eval_env, prefix="[benchmark] ")
                BENCH_OK = BENCH_RECORD.is_file()
                PHASE_STATUS["benchmark"] = "passed" if BENCH_OK else "failed_no_record"
            except (RuntimeError, OSError, ValueError) as exc:
                PHASE_STATUS["benchmark"] = f"failed: {exc}"
                print("Benchmark failed:", exc, flush=True)
        else:
            PHASE_STATUS["benchmark"] = "skipped_calibration_failed"
            print("Skipping benchmark because calibration did not produce a valid contract.", flush=True)

## 8. Review paired results and save artifacts

The JSON record contains per-question correctness and aggregate strict/lenient accuracy. This cell prints the configured-test-set lenient scores and the paired steer-only-minus-baseline delta for each shot condition.

In [ ]:
if SESSION == 1:
    if BENCH_OK:
        record_path = BENCH_RECORD
        record = json.loads(record_path.read_text(encoding="utf-8"))
        summary = {}
        for shots, result in record["shots"].items():
            baseline = result["baseline"]
            steer_only = result["steer_only"]
            summary[shots] = {
                "n": baseline["n"],
                "baseline_lenient_accuracy": baseline["lenient_accuracy"],
                "steer_only_lenient_accuracy": steer_only["lenient_accuracy"],
                "paired_delta_pp": steer_only["lenient_accuracy"] - baseline["lenient_accuracy"],
                "expected_n": BENCH_N_TEST,
                "n_matches_config": baseline["n"] == steer_only["n"] == BENCH_N_TEST,
            }
        print(json.dumps(summary, indent=2))
        print("Artifacts:", record_path, Path("checkpoints/steernet.pt"), CONTRACT_PATH)
    else:
        print("No benchmark record was produced; see phase statuses for the skip/failure reason.", flush=True)
    print("Session 1 phase status:")
    print(json.dumps(PHASE_STATUS, indent=2))